# TP 15 — Méthodes d'ensemble

Données : © Open Food Facts contributors — ODbL

In [2]:
import pandas as pd

from nutriscope.ml.data import LABELS, column_types, load_dataset, split_xy
from nutriscope.ml.models import build_pipeline, tp14_models, boosting_models
from nutriscope.ml.protocol import CROSS_VALIDATION, SCORING, evaluate_cross_validation

#### Chargement des données pour le TP

In [3]:
dataframe = load_dataset()
X_train, _, y_train, _ = split_xy(dataframe)
numeric_columns, categorical_columns = column_types(X_train)

print(X_train.shape)

(24076, 13)


#### Réentrainement dans les conditions imposées pour le TP

In [4]:
models = tp14_models()
names = [
    'logreg',
    'random_forest',
    'random_forest_balanced',
]

results_tp14 = [
    evaluate_cross_validation(
        build_pipeline(models[name], numeric_columns, categorical_columns),
        X_train,
        y_train,
        name,
    )
    for name in names
]

table_tp14 = pd.DataFrame(results_tp14).set_index('modele')
table_tp14.round(4)

,f1_macro_moy,f1_macro_std,accuracy_moy,accuracy_std,balanced_accuracy_moy,balanced_accuracy_std,fit_time_moy,score_time_moy
modele,,,,,,,,
logreg,0.7637,0.0080,0.8016,0.0055,0.7624,0.0075,0.9073,0.0310
random_forest,0.8551,0.0071,0.8751,0.0059,0.8508,0.0065,4.4578,0.1902
random_forest_balanced,0.8585,0.0047,0.8754,0.0041,0.8586,0.0043,4.6739,0.1910


#### Point de contrôle 1

In [5]:
# F1 macro (CV) lus dans le tableau du TP 14
REFERENCE_TP14 = {
    'logreg': 0.7605,
    'random_forest': 0.8530,
    'random_forest_balanced': 0.858,
}
THRESHOLD = 0.01

comparison = pd.DataFrame({
    'TP 14': pd.Series(REFERENCE_TP14),
    'TP 15': table_tp14.loc[list(REFERENCE_TP14), 'f1_macro_moy'],
})
comparison['écart'] = (comparison['TP 15'] - comparison['TP 14']).abs()
comparison['statut'] = comparison['écart'].apply(
    lambda gap: 'OK' if gap <= THRESHOLD else 'À VÉRIFIER'
)

display(
    comparison.style
    .format({
        'TP 14': '{:.4f}',
        'TP 15': '{:.4f}',
        'écart': '{:.4f}',
    })
    .map(
        lambda value: 'color: green' if value == 'OK' else 'color: red',
        subset=['statut'],
    )
)

assert (comparison['écart'] <= THRESHOLD).all(), (
    f'Écart supérieur à {THRESHOLD} : la préparation a bougé ?'
)

,TP 14,TP 15,écart,statut
logreg,0.7605,0.7637,0.0032,OK
random_forest,0.8530,0.8551,0.0021,OK
random_forest_balanced,0.8580,0.8585,0.0005,OK


Les trois modèles du TP 14 sont retrouvés à moins de 0,004 près (seuil du sujet : 0,01).
L'écart reste inférieur à l'écart-type des plis : il s'explique par le passage du split
recalculé à la colonne `split` figée du TP 13, pas par un changement de préparation.
La comparaison avec le boosting peut donc se faire sur cette base.

In [ ]:
label_to_integer = {label: index for index, label in enumerate(LABELS)}
y_train_integer = y_train.map(label_to_integer)

results_boosting = [
    evaluate_cross_validation(
        build_pipeline(model, numeric_columns, categorical_columns),
        X_train,
        y_train_integer,
        name,
    )
    for name, model in boosting_models().items()
]

table_boosting = pd.DataFrame(results_boosting).set_index('modele')
table_boosting.round(4)

,f1_macro_moy,f1_macro_std,accuracy_moy,accuracy_std,balanced_accuracy_moy,balanced_accuracy_std,fit_time_moy,score_time_moy
modele,,,,,,,,
hist_gradient_boosting,0.8895,0.0038,0.9075,0.0037,0.8879,0.0036,5.1211,0.1346
xgboost,0.8946,0.0053,0.9112,0.0037,0.8932,0.0053,7.2191,0.1168
lightgbm,0.8945,0.0097,0.9116,0.0076,0.8937,0.0094,3.7087,0.5241


In [7]:
reference_f1 = table_tp14.loc['random_forest_balanced', 'f1_macro_moy']

summary = pd.DataFrame({
    'f1_macro': table_boosting['f1_macro_moy'],
    'écart-type': table_boosting['f1_macro_std'],
})
summary['gain vs forêt équilibrée'] = summary['f1_macro'] - reference_f1
summary['gain en écarts-types'] = summary['gain vs forêt équilibrée'] / summary['écart-type']

summary.round(4)

,f1_macro,écart-type,gain vs forêt équilibrée,gain en écarts-types
modele,,,,
hist_gradient_boosting,0.8895,0.0038,0.0310,8.2360
xgboost,0.8946,0.0053,0.0361,6.8362
lightgbm,0.8945,0.0097,0.0359,3.7072
